<a href="https://colab.research.google.com/github/SURENAANERUS/Projects/blob/main/model19.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive') # import the data from google Drive

In [ ]:
import numpy as np
import torch

b = np.load('/content/drive/MyDrive/chess_dataset_0 (3).npz') # load the data


In [ ]:
xNp = b['X']
yNp = b['y']
idsNp = b['game_ids']

x = torch.from_numpy(xNp)
y = torch.from_numpy(yNp)
ids = torch.from_numpy(idsNp)


# get rid of the useless matrices

x = x[:,:16]
print(x.shape)

# free ram

import gc

del xNp, yNp, idsNp  # delete your original numpy arrays
gc.collect()    # force garbage collection

In [ ]:

use_cuda = torch.cuda.is_available()
device = "cuda" if use_cuda else "cpu"

trainRatio = 0.7

trainLen = int(len(x) * trainRatio)
xTrain,xTest = x[:trainLen].float(), x[trainLen:].float()
yTrain, yTest = y[:trainLen].float(), y[trainLen:].float()


xTrain = xTrain.to(device)
yTrain = yTrain.to(device)

xTest = xTest.to(device)
yTest = yTest.to(device)


print("devices of data:", xTrain.device, xTest.device, yTrain.device, yTest.device)


trainData = torch.utils.data.TensorDataset(xTrain,yTrain)
testData = torch.utils.data.TensorDataset(xTest,yTest)





#print(x.shape, y.shape, ids.shape) # these are  8024029 x 18 x 8 x 8, so 8024029 instances, of 18 tensors, 8x8 boards.
bs = 4096 #
trainDL = torch.utils.data.DataLoader(trainData,bs)
testDL = torch.utils.data.DataLoader(testData,bs)




In [9]:
import torch.nn as nn
# now it's time to create the network itself.
# we start with B x 16 x 8 x 8
class Model(nn.Module):
  def __init__(self):
    super().__init__()
    self.reLu = nn.ReLU()
    self.bn0 = nn.BatchNorm2d(16)

    self.conv1 = nn.Conv2d(16,32,3,1,padding="same") # now its 32 x 8 x 8
    self.bn1 = nn.BatchNorm2d(32)

    self.conv2 = nn.Conv2d(32,64,3,1,padding="same") # now its  64
    self.bn2 = nn.BatchNorm2d(64)

    self.conv3 = nn.Conv2d(64,128,3,1,padding="same") # now its 128
    self.bn3 = nn.BatchNorm2d(128)

    self.ln = nn.Linear(8192,1)

  def forward(self, x: torch.Tensor) -> torch.Tensor:
    # x comes as B x 16 x 8 x 8
    #x = self.bn0(x)
    #print("enter 1 block")
    x = self.conv1(x)
    x = self.bn1(x)
    x = self.reLu(x)

    #print("enter 2 block")

    x = self.conv2(x)
    x = self.bn2(x)
    x = self.reLu(x)
    #print("enter 3 block")
    x = self.conv3(x)
    x = self.bn3(x)
    x = self.reLu(x) # 128 x 8 x 8



    x = x.view(-1,128 * 8 * 8) #
    #print("NEW SHAPE: ", x.shape)
    x = self.ln(x)
    x = x.squeeze(1)
    #print("final x shape: ", x.shape)
    return x

model = Model().to(device)
# now onto the train and test loops



In [10]:
import torch.nn.functional as F
def train_epoch(
    model: torch.nn.Module,
    device: torch.device,
    trainLoader: torch.utils.data.DataLoader,
    opt: torch.optim.Optimizer,
    epoch:int,
    log_interval: int,
) -> None:
  model.train()
  for batchIdx, (data, target) in enumerate(trainLoader):
    opt.zero_grad()
    output = model(data)
    loss = F.binary_cross_entropy_with_logits(output,target)

    loss.backward()
    opt.step()

    if batchIdx % log_interval == 0:
      print(f"Train Epoch: {epoch}, training loss: {loss.item():.6f}")

def test(
  model: torch.nn.Module,
  device: torch.device,
  testLoader: torch.utils.data.DataLoader,
  epoch:int,
) -> None:
  model.eval()
  testLoss = 0
  nCorrect = 0
  nTotal = 0
  batchNum = 0
  with torch.no_grad():
    for data,target in testLoader:

      output = model(data)
      testLoss += F.binary_cross_entropy_with_logits(output,target)
      probs = torch.sigmoid(output) # we need probabilities, not logits
      preds = (probs > 0.5).float()

      nCorrect += (preds == target).sum().item()
      nTotal += len(target)
      batchNum += 1 # loss jest akumulowany per batch, wiec dzierlimy prze ilosc batchy
    testLoss /= batchNum
    accuracy = nCorrect / nTotal

    print(f"Test loss: {testLoss:.4f}, Accuracy: {accuracy}")
      # now preds should be B x 1, with True False True false


In [11]:
# Let it rip
EPOCHS = 30
optimizer = torch.optim.Adam(model.parameters()) # defcault lr is 0.01 IIRC

for epoch in range(EPOCHS):
  train_epoch(model,device,trainDL,optimizer,epoch,10000)
  print("train done!")
  test(model,device,testDL,epoch)
  print("test done! ")



Train Epoch: 0, training loss: 0.745985
train done!
Test loss: 0.6186, Accuracy: 0.6178611828054814
test done! 
Train Epoch: 1, training loss: 0.610358
train done!
Test loss: 0.6114, Accuracy: 0.6250674536361405
test done! 
Train Epoch: 2, training loss: 0.604120
train done!
Test loss: 0.6090, Accuracy: 0.6284377467847619
test done! 
Train Epoch: 3, training loss: 0.605489
train done!
Test loss: 0.6045, Accuracy: 0.6337139816276859
test done! 
Train Epoch: 4, training loss: 0.604444
train done!
Test loss: 0.6025, Accuracy: 0.636636868672392
test done! 
Train Epoch: 5, training loss: 0.602100
train done!
Test loss: 0.6009, Accuracy: 0.6379367142612046
test done! 
Train Epoch: 6, training loss: 0.599168
train done!
Test loss: 0.5995, Accuracy: 0.6394662864753331
test done! 
Train Epoch: 7, training loss: 0.596022
train done!
Test loss: 0.5992, Accuracy: 0.6400387336537874
test done! 
Train Epoch: 8, training loss: 0.592227
train done!
Test loss: 0.5989, Accuracy: 0.6406909412518813
test 

KeyboardInterrupt: 